# Cleaning


Ziel dieses Notebooks ist es, den importierten Datensatz gezielt für die Beantwortung der folgenden Businessfrage aufzubereiten.

## Businessfrage

**Wie entwickeln sich Umsatz, Absatz und Bestellstruktur regulärer Produktverkäufe im Zeitverlauf, und welche Produkte und Länder treiben diese Entwicklung?**

Die Businessfrage richtet sich ausschließlich auf reguläre Produktverkäufe. Analysiert werden sollen unter anderem die Entwicklung von Bruttoumsatz,
Absatz, Bestellwert und Bestellstruktur sowie Unterschiede zwischen Produkten und Ländern. Der daraus entstehende Datensatz beschreibt somit die
ursprüngliche Nachfrage nach Produkten.

Im Data Profiling wurde festgestellt, dass der Datensatz neben regulären Produktverkäufen auch vollständige Duplikate, Stornierungen, administrative
Vorgänge und weitere Sonderfälle enthält. Diese Datensätze sind nicht grundsätzlich fehlerhaft und können für andere Fragestellungen durchaus relevant sein.
Für die vorliegende Businessfrage liefern sie jedoch keinen fachlichen Mehrwert oder würden die Aussagekraft der späteren Analysen beeinflussen.

Die folgenden Cleaning-Schritte orientieren sich daher nicht an einer allgemeinen Datenbereinigung, sondern an den Anforderungen der Businessfrage.

Jede Änderung am Datensatz wird fachlich begründet, umgesetzt und anschließend überprüft.

In [1]:
from src.paths import IMPORTED_DATA_DIR, CLEANED_DATA_DIR
import pandas as pd

dataset = pd.read_parquet(IMPORTED_DATA_DIR / "online_retail_II.parquet")

## Entfernen vollständiger Duplikate

### Businessbezug

Die Businessfrage untersucht die Entwicklung von Bruttoumsatz, Absatz und Bestellstruktur regulärer Produktverkäufe. Jede Transaktion darf daher nur einmal in die Analyse eingehen.

### Begründung

Im Data Profiling wurde festgestellt, dass der Datensatz vollständig identische Datensätze enthält. Da sich diese Datensätze in keiner Spalte unterscheiden, repräsentieren sie keinen eigenständigen Geschäftsvorfall, sondern dieselbe Transaktion mehrfach.

Würden diese Duplikate im Datensatz verbleiben, würden Umsatz, Absatz und Bestellstruktur mehrfach berücksichtigt und die Ergebnisse der späteren Analysen verfälschen.

### Einschränkung

Durch das Entfernen vollständiger Duplikate geht keine fachliche Information verloren, da ausschließlich redundante Datensätze entfernt werden.

### Cleaning Code

#### Vollständige Duplikate identifizieren

Zunächst wird überprüft, ob der importierte Datensatz die im Data Profiling ermittelte Anzahl an Zeilen enthält. Anschließend werden alle vollständig
identischen Datensätze als Duplikate identifiziert.

In [2]:
expected_rows = 1_067_371

assert dataset.shape[0] == expected_rows, (
    "The imported dataset does not match the expected number of rows."
)

dataset = dataset.drop_duplicates()

rows_after = dataset.shape[0]

#### Ergebnis validieren

Die Assertions prüfen, ob dieselbe Anzahl an Duplikaten entfernt wurde wie im Data Profiling ermittelt.

In [3]:
expected_duplicates = 34_335

assert rows_after == expected_rows - expected_duplicates, (
    "Unexpected number of duplicate rows removed."
)

print(f"Rows before cleaning : {expected_rows:,}")
print(f"Removed duplicate rows: {expected_duplicates:,}")
print(f"Rows after cleaning  : {rows_after:,}")

Rows before cleaning : 1,067,371
Removed duplicate rows: 34,335
Rows after cleaning  : 1,033,036


Die Anzahl der entfernten Datensätze stimmt mit den Ergebnissen des Data Profilings überein.
Der Datensatz enthält keine vollständig identischen Datensätze mehr.

## Entfernen von Stornierungen

### Businessbezug

Die Businessfrage untersucht die Entwicklung von Bruttoumsatz, Absatz und Bestellstruktur regulärer Produktverkäufe.
Dafür soll der Datensatz ausschließlich ursprüngliche Produktbestellungen enthalten.

### Begründung

Laut der Dokumentation des Datensatzes kennzeichnet eine **Invoice**, die mit `C` beginnt, eine Stornierung.
Eine Stornierung beschreibt die vollständige oder teilweise Rücknahme einer zuvor erfassten Bestellung und stellt somit keine neue Kaufentscheidung dar.

Für die vorliegende Businessfrage ist die ursprüngliche Nachfrage nach Produkten entscheidend.
Stornierungen werden daher bewusst ausgeschlossen, da sie keine zusätzlichen Informationen über das Kaufverhalten liefern, sondern bereits erfasste
Bestellungen korrigieren oder aufheben würden.

### Welche Aussage wird dadurch möglich?

Der bereinigte Datensatz beschreibt die Entwicklung der ursprünglichen Produktbestellungen.
Dadurch lassen sich Nachfrage, Bruttoumsatz, Absatz und Bestellstruktur im Zeitverlauf sowie Unterschiede zwischen Produkten und Ländern untersuchen.

### Einschränkung

Der Datensatz eignet sich nach diesem Schritt nicht mehr zur Analyse von Nettoerlösen, tatsächlich verkauften Mengen oder Stornoquoten,
da die Rücknahme von Bestellungen nicht mehr berücksichtigt wird.

### Cleaning Code

#### Stornorechnungen identifizieren

Stornorechnungen werden anhand des Präfixes `C` in der Rechnungsnummer identifiziert.

In [4]:
expected_rows = rows_after

cancellation_mask = dataset["Invoice"].str.startswith("C")

#### Stornorechnungen entfernen

Alle identifizierten Stornorechnungen werden aus dem Analysebestand entfernt.

In [5]:
dataset = dataset.drop(dataset[cancellation_mask].index)

rows_after = dataset.shape[0]

#### Ergebnis validieren

Die Assertions prüfen, ob dieselbe Anzahl an Stornorechnungen entfernt wurde wie im Data Profiling ermittelt. Zusätzlich wird überprüft, ob
keine Stornorechnungen mehr im Datensatz vorhanden sind.

In [6]:
expected_cancellations = 19_104

assert rows_after == expected_rows - expected_cancellations, (
    "Unexpected number of cancellation rows removed."
)

has_cancellation = dataset["Invoice"].str.startswith("C").any()

assert not has_cancellation, (
   "Cancellation rows are still present in the dataset."
)

print(f"Rows before cleaning : {expected_rows:,}")
print(f"Removed cancellation rows: {expected_cancellations:,}")
print(f"Rows after cleaning  : {rows_after:,}")

Rows before cleaning : 1,033,036
Removed cancellation rows: 19,104
Rows after cleaning  : 1,013,932


## Vereinheitlichen des Rechnungszeitpunkts

### Begründung

Im Data Profiling wurde festgestellt, dass einzelne Rechnungen geringfügig unterschiedliche Zeitstempel aufweisen. Die durchschnittliche Abweichung beträgt rund
eine Minute, die maximale Abweichung neun Minuten.

Da alle Positionen derselben **Invoice** zu einer Rechnung gehören, sollen sie auch einen gemeinsamen Rechnungszeitpunkt besitzen. Der früheste vorhandene
Zeitstempel wird dafür als Rechnungszeitpunkt verwendet, da er den Zeitpunkt repräsentiert, zu dem die Rechnung im Datensatz erstmals erfasst wurde.
Spätere Zeitstempel derselben Rechnung liegen zeitlich danach und eignen sich daher nicht als ursprünglicher Rechnungszeitpunkt.

### Cleaning Code

#### Rechnungszeitpunkt vereinheitlichen

In [7]:
dataset["InvoiceDate"] = (
    dataset.groupby("Invoice")["InvoiceDate"]
    .transform("min")
)

## Auswahl regulärer Produktverkäufe

### Businessbezug

Nach dem Entfernen der Stornierungen enthält der Datensatz weiterhin Sonderfälle, die keine regulären Produktverkäufe beschreiben.
Dazu gehören unter anderem administrative Vorgänge, Gutscheine, Testbuchungen sowie Transaktionen mit nicht positiven Mengen oder Preisen.

Für die Businessfrage werden ausschließlich reguläre Produktverkäufe benötigt, da nur diese Aussagen über die Entwicklung von
Bruttoumsatz, Absatz und Bestellstruktur ermöglichen.

### Begründung

Im Data Profiling wurden alle Sonderfälle untersucht und fachlich bewertet. Dabei zeigte sich, dass bestimmte **StockCodes** keine
regulären Produkte repräsentieren, sondern beispielsweise Versandkosten, Rabatte, Gebühren, manuelle Anpassungen, Gutscheine oder
Testbuchungen.

Darüber hinaus wurden die verbleibenden Datensätze mit negativer **Quantity** untersucht. Die Ergebnisse des Profilings sprechen dafür,
dass diese Datensätze überwiegend interne Bestands- und Korrekturvorgänge beschreiben und keine regulären Produktverkäufe darstellen.
Sie werden daher ebenfalls nicht in den Analysebestand übernommen.

### Welche Aussage wird dadurch möglich?

Der verbleibende Datensatz beschreibt ausschließlich reguläre Produktverkäufe. Dadurch können Bruttoumsatz, Absatz und Bestellstruktur
im Zeitverlauf analysiert sowie Produkte und Länder identifiziert werden, die die Entwicklung der ursprünglichen Produktnachfrage
bestimmen.

### Cleaning Code


#### Zu entfernende Datensätze identifizieren

Zunächst werden alle Datensätze identifiziert, die nicht Bestandteil des Analysebestands sind. Dazu zählen administrative Vorgänge, Gutscheine, Testprodukte, manuelle Anpassungen sowie
Datensätze mit einer negativen Menge.

In [8]:
expected_rows = rows_after

dataset["StockCode"] = dataset["StockCode"].str.strip()

not_regular_stockcodes = [
    "BANK CHARGES", # Bank charges
    "AMAZONFEE",    # Amazon fee
    "POST",         # Postage
    "CRUK",         # Cancer Research UK commission
    "DOT",          # Dotcom postage
    "C2",           # Carriage
    "C3",           # Carriage
    "B",            # Bad debt adjustment
    "D",            # Discount
    "m",            # Manual
    "M",            # Manual
    "S"             # Samples
]

drop_mask = dataset["StockCode"].isin(not_regular_stockcodes)

stock_codes = dataset["StockCode"].str.lower()

drop_mask |= (
    stock_codes.str.startswith("gift")       # Gift voucher
    | stock_codes.str.startswith("test")     # Test product
    | stock_codes.str.startswith("adjust")   # Manual adjustment
)

drop_mask |= (dataset["Quantity"] < 0)

#### Identifizierte Datensätze entfernen

Alle identifizierten Datensätze werden gemeinsam aus dem Analysebestand entfernt.

In [9]:
dataset = dataset.drop(dataset[drop_mask].index)

In [10]:
rows_after = dataset.shape[0]

print(f"Rows before cleaning : {expected_rows:,}")
print(f"Rows removed         : {expected_rows - rows_after:,}")
print(f"Rows after cleaning  : {rows_after:,}")

Rows before cleaning : 1,013,932
Rows removed         : 8,003
Rows after cleaning  : 1,005,929


## Nullpreis-Rechnungen

### Businessbezug

Für die Businessfrage werden ausschließlich reguläre Produktverkäufe betrachtet. Rechnungen, die ausschließlich aus kostenlosen Positionen bestehen, können sowohl interne
Geschäftsvorgänge als auch kostenlose Produktabgaben enthalten und werden deshalb näher untersucht.

### Begründung

Im Data Profiling wurden alle Rechnungen untersucht, die ausschließlich Positionen mit einem Preis von `0` enthalten.

Dabei zeigte sich, dass ein Teil der Rechnungen mit Nullpreispositionen zusätzlich kostenpflichtige Positionen enthält. Diese Rechnungen bleiben unverändert im Analysebestand,
da die kostenlosen Positionen Teil einer regulären Kundenbestellung sein können.

Die verbleibenden reinen Nullpreis-Rechnungen wurden anschließend anhand ihrer Produktbeschreibungen untersucht. Datensätze ohne Produktbeschreibung sowie Datensätze mit eindeutig
internen Beschreibungen (z. B. Inventur-, Prüf- oder Korrekturhinweise) werden ausgeschlossen. Reguläre Produktbezeichnungen bleiben hingegen erhalten, da sie tatsächliche Produktabgaben
beschreiben können und keine belastbaren Hinweise auf interne Vorgänge vorliegen.

### Welche Aussage wird dadurch möglich?

Der Analysebestand enthält reguläre Produktabgaben, während eindeutig interne Nullpreis-Buchungen ausgeschlossen werden. Dadurch bleiben Produktanalysen vollständig erhalten,
ohne interne Verwaltungs- oder Korrekturvorgänge in die Auswertung einzubeziehen.

## Cleaning Code

#### Rechnungen mit Nullpreispositionen identifizieren

Zunächst werden alle Rechnungen ermittelt, die mindestens eine Position mit einem Preis von 0 enthalten. Anschließend wird zwischen gemischten Rechnungen und Rechnungen unterschieden, die ausschließlich aus Nullpreispositionen bestehen.

In [11]:
expected_rows = rows_after

price_zero = dataset[dataset["Price"] == 0].drop_duplicates(subset="Invoice")

#### Rechnungen mit ausschließlich kostenlosen Positionen auswählen

Gemischte Rechnungen bleiben unverändert, da kostenlose Positionen Teil einer regulären Kundenbestellung sein können. Für die weitere Untersuchung werden ausschließlich Rechnungen betrachtet,
deren Positionen vollständig kostenlos sind.

In [12]:
invoice_rows = dataset[dataset["Invoice"].isin(price_zero["Invoice"])]

free_invoice_summary = (
    invoice_rows.groupby("Invoice", as_index=False)["Price"]
    .agg(is_free_invoice=lambda grp: (grp == 0).all())
)

free_invoices = free_invoice_summary.loc[free_invoice_summary["is_free_invoice"], "Invoice"]

free_invoice_rows = invoice_rows[invoice_rows["Invoice"].isin(free_invoices)]

print(f"Free invoices: {free_invoices.nunique():,}")
print(f"Rows in free invoices: {free_invoice_rows.shape[0]:,}")

Free invoices: 1,842
Rows in free invoices: 2,446


#### Datensätze ohne Produktbeschreibung entfernen

Datensätze ohne Produktbeschreibung liefern keine ausreichenden Informationen für die Analyse regulärer Produktverkäufe und werden deshalb entfernt.

In [13]:
missing_description_mask = free_invoice_rows["Description"].isna()

dataset = dataset.drop(missing_description_mask[missing_description_mask].index)

free_invoice_rows = free_invoice_rows.dropna(subset="Description")

print(f"Removed rows without Description: {missing_description_mask.sum():,}")

Removed rows without Description: 1,608


#### Interne Beschreibungen identifizieren

Die verbleibenden Beschreibungen werden vereinheitlicht und mit den im Profiling identifizierten internen Bezeichnungen verglichen.

In [14]:
description = free_invoice_rows["Description"].str.strip().str.lower()

drop_mask = (
    description.str.startswith("?sold")                # ?sold individually?
    | description.str.startswith("add stock")          # add stock to allocate online orders
    | description.str.startswith("adjust")             # adjustment
    | description.str.startswith("allocate stock")     # allocate stock for dotcom orders
    | description.str.startswith("amazon")             # amazon, amazon sales, amazon adjustment
    | description.str.startswith("check")              # check, checked, check?
    | description.str.startswith("dotcom")             # dotcom, dotcom email, dotcom adjust, dotcomstock
    | description.str.startswith("found")              # found, found again, found box, found in w/hse
    | description.str.startswith("incorrect")          # incorrectly credited
    | description.str.startswith("mailout")            # mailout, mailout addition
    | description.str.startswith("marked as")          # marked as 23343
    | description.str.startswith("sold ")              # sold as a/b
    | description.str.startswith("wrong")              # wrong invc, wrongly coded, wrongly marked
)

exact_descriptions = [
    "?",
    "22719",
    "alan hodge cant mamage this section",
    "amendment",
    "came coded as 20713",
    "correct previous adjustment",
    "damaged",
    "debenhams",
    "did  a credit  and did not tick ret",
    "eurobargain invc/credit",
    "fba",
    "for online retail orders",
    "had been put aside",
    "had been put aside.",
    "john lewis",
    "lighthouse trading zero invc incorr",
    "michel oops",
    "on cargo order",
    "rcvd be air temp fix for dotcom sit",
    "returned",
    "rust fixed",
    "sale error",
    "taig adjust",
    "temp",
    "test",
    "tk maxx mix up with pink",
    "update",
    "website fixed"
]

drop_mask |= description.isin(exact_descriptions)

#### Interne Geschäftsvorgänge entfernen

Datensätze mit eindeutig internen Beschreibungen werden aus dem Analysebestand entfernt.

In [15]:
internal_description_rows = drop_mask.sum()

dataset = dataset.drop(drop_mask[drop_mask].index)

print(f"Removed rows with internal descriptions: {internal_description_rows:,}")

Removed rows with internal descriptions: 203


In [16]:
rows_after = dataset.shape[0]

print(f"Rows before cleaning : {expected_rows:,}")
print(f"Removed rows         : {expected_rows - rows_after:,}")
print(f"Rows after cleaning  : {rows_after:,}")

Rows before cleaning : 1,005,929
Removed rows         : 1,811
Rows after cleaning  : 1,004,118


## Speichern des gesäuberten Datensatzes

Der gesäuberte Datensatz wird im `Parquet`-Format gespeichert. Die erzeugte Datei dient als Eingabe für die Transformation.

In [17]:
dataset.to_parquet(CLEANED_DATA_DIR / "online_retail_II.parquet")